In [20]:
# Cell 1 — Imports
import pandas as pd
import numpy as np
from datetime import datetime

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)


In [21]:
# Cell 2 — Load the raw dataset
import pandas as pd

RAW_FILE = r"C:\Users\SAIM\Downloads\Dataset for Data Analytics.xlsx"

df_raw = pd.read_excel(RAW_FILE)

# Keep an untouched copy for before/after comparison in the change log
df = df_raw.copy()

print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
df.head()

Rows: 1200, Columns: 14


,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


In [22]:
# Cell 3 — Initial inspection: structure, dtypes, and a quick look
df.info()
print()
print("Column list:", list(df.columns))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   OrderID          1200 non-null   object        
 1   Date             1200 non-null   datetime64[ns]
 2   CustomerID       1200 non-null   object        
 3   Product          1200 non-null   object        
 4   Quantity         1200 non-null   int64         
 5   UnitPrice        1200 non-null   float64       
 6   ShippingAddress  1200 non-null   object        
 7   PaymentMethod    1200 non-null   object        
 8   OrderStatus      1200 non-null   object        
 9   TrackingNumber   1200 non-null   object        
 10  ItemsInCart      1200 non-null   int64         
 11  CouponCode       891 non-null    object        
 12  ReferralSource   1200 non-null   object        
 13  TotalPrice       1200 non-null   float64       
dtypes: datetime64[ns](1), float64(2), int64(

In [23]:
# Cell 4 — Identify missing / null values
missing_summary = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_pct": (df.isnull().sum() / len(df) * 100).round(2)
})
missing_summary = missing_summary[missing_summary["missing_count"] > 0].sort_values(
    "missing_count", ascending=False
)
print("Columns with missing values:")
missing_summary


Columns with missing values:


,missing_count,missing_pct
CouponCode,309,25.75


In [24]:
# Cell 5 — Impute missing values (Mean / Median / Mode strategy)
change_log = []  # we will build our official "PDF Change Log" from this list

def log_change(change_id, description, impact, status="Resolved"):
    change_log.append({
        "Change ID": change_id,
        "Description": description,
        "Impact": impact,
        "Status": status
    })

# --- Categorical: CouponCode (NaN legitimately means "no coupon was applied") ---
if "CouponCode" in df.columns:
    n_missing = df["CouponCode"].isnull().sum()
    if n_missing > 0:
        df["CouponCode"] = df["CouponCode"].fillna("No Coupon")
        log_change(
            "CR001",
            "Imputed 'CouponCode' missing values with explicit label 'No Coupon'",
            f"Preserved {n_missing} records (avoided deletion)"
        )

# --- Numeric columns: median imputation (robust to outliers) ---
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    n_missing = df[col].isnull().sum()
    if n_missing > 0:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        log_change(
            f"CR_{col}",
            f"Imputed '{col}' missing values using MEDIAN ({median_val})",
            f"Preserved {n_missing} records"
        )

# --- Remaining categorical/text columns: mode imputation ---
categorical_cols = df.select_dtypes(include=["object"]).columns
for col in categorical_cols:
    n_missing = df[col].isnull().sum()
    if n_missing > 0:
        mode_val = df[col].mode(dropna=True)[0]
        df[col] = df[col].fillna(mode_val)
        log_change(
            f"CR_{col}",
            f"Imputed '{col}' missing values using MODE ('{mode_val}')",
            f"Preserved {n_missing} records"
        )

print(f"Total missing values remaining: {df.isnull().sum().sum()}")


Total missing values remaining: 0


In [25]:
# Cell 6 — Duplicate audit (full-row duplicates + duplicate unique IDs)
full_dupe_count = df.duplicated().sum()
orderid_dupe_count = df["OrderID"].duplicated().sum() if "OrderID" in df.columns else None
tracking_dupe_count = df["TrackingNumber"].duplicated().sum() if "TrackingNumber" in df.columns else None

print(f"Full duplicate rows: {full_dupe_count}")
print(f"Duplicate OrderID values: {orderid_dupe_count}")
print(f"Duplicate TrackingNumber values: {tracking_dupe_count}")

# Equivalent SQL-style check (GROUP BY OrderID HAVING COUNT(*) > 1)
dupe_orderids = (
    df.groupby("OrderID").size().reset_index(name="count").query("count > 1")
)
dupe_orderids


Full duplicate rows: 0
Duplicate OrderID values: 0
Duplicate TrackingNumber values: 0


,OrderID,count


In [26]:
# Cell 7 — Remove duplicates
before_rows = len(df)

# 1) Drop fully duplicated rows outright
df = df.drop_duplicates()

# 2) For duplicate OrderIDs (same unique identifier, potentially differing rows),
#    keep only the first occurrence — the canonical record
df = df.drop_duplicates(subset="OrderID", keep="first")

after_rows = len(df)
rows_removed = before_rows - after_rows

log_change(
    "CR_DEDUPE",
    "Removed full-row duplicates and collapsed duplicate OrderID records to a single canonical row",
    f"Removed {rows_removed} duplicate record(s); {after_rows} unique records remain",
)

print(f"Rows before: {before_rows} | Rows after: {after_rows} | Removed: {rows_removed}")


Rows before: 1200 | Rows after: 1200 | Removed: 0


In [27]:
# Cell 8 — Standardize text columns (trim whitespace + consistent case)
text_cols = df.select_dtypes(include=["object"]).columns
whitespace_fixed = 0

for col in text_cols:
    original = df[col].copy()
    df[col] = df[col].astype(str).str.strip()
    whitespace_fixed += (original.astype(str) != df[col]).sum()

# Title-case free-text / categorical columns (skip pure ID codes)
id_like_cols = {"OrderID", "CustomerID", "TrackingNumber", "CouponCode"}
for col in text_cols:
    if col not in id_like_cols:
        df[col] = df[col].str.title()

log_change(
    "CR_TEXT_STD",
    "Trimmed whitespace and applied consistent Title Case across text/categorical columns",
    f"Corrected formatting on {whitespace_fixed} cell(s)"
)

print("Sample standardized categories:")
for col in ["Product", "PaymentMethod", "OrderStatus", "ReferralSource"]:
    if col in df.columns:
        print(f"  {col}: {sorted(df[col].unique())}")


Sample standardized categories:
  Product: ['Chair', 'Desk', 'Laptop', 'Monitor', 'Phone', 'Printer', 'Tablet']
  PaymentMethod: ['Cash', 'Credit Card', 'Debit Card', 'Gift Card', 'Online']
  OrderStatus: ['Cancelled', 'Delivered', 'Pending', 'Returned', 'Shipped']
  ReferralSource: ['Email', 'Facebook', 'Google', 'Instagram', 'Referral']


In [28]:
# Cell 9 — Standardize dates to ISO 8601 (YYYY-MM-DD)
if "Date" in df.columns:
    before_bad_dates = df["Date"].isna().sum()
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    after_bad_dates = df["Date"].isna().sum()

    # Store as proper ISO 8601 string for the final "gold standard" export
    df["Date"] = df["Date"].dt.strftime("%Y-%m-%d")

    log_change(
        "CR_DATE_ISO",
        "Converted 'Date' column to ISO 8601 format (YYYY-MM-DD)",
        f"Standardized {len(df)} records; {after_bad_dates} unparseable date(s) flagged"
    )

df[["Date"]].head() if "Date" in df.columns else None


,Date
0,2023-01-04
1,2024-08-23
2,2024-02-27
3,2023-10-15
4,2025-05-08


In [29]:
# Cell 10 — Enforce numeric precision (2 decimal places) on monetary/decimal fields
money_cols = [c for c in ["UnitPrice", "TotalPrice"] if c in df.columns]

for col in money_cols:
    df[col] = df[col].astype(float).round(2)

if money_cols:
    log_change(
        "CR_NUMERIC_PRECISION",
        f"Rounded {', '.join(money_cols)} to 2 decimal places for numeric consistency",
        f"Applied to {len(df)} records"
    )

df[money_cols].head() if money_cols else None


,UnitPrice,TotalPrice
0,570.62,2853.10
1,151.35,302.70
2,550.68,2753.40
3,273.19,273.19
4,626.01,2504.04


In [30]:
# Cell 11 — Verify TotalPrice = Quantity x UnitPrice
if {"Quantity", "UnitPrice", "TotalPrice"}.issubset(df.columns):
    expected_total = (df["Quantity"] * df["UnitPrice"]).round(2)
    mismatch_mask = (expected_total - df["TotalPrice"]).abs() > 0.01
    n_mismatch = mismatch_mask.sum()

    if n_mismatch > 0:
        df.loc[mismatch_mask, "TotalPrice"] = expected_total[mismatch_mask]
        log_change(
            "CR_LOGIC_TOTALPRICE",
            "Recalculated 'TotalPrice' where it did not equal Quantity x UnitPrice",
            f"Corrected {n_mismatch} record(s)"
        )
    else:
        log_change(
            "CR_LOGIC_TOTALPRICE",
            "Verified 'TotalPrice' equals Quantity x UnitPrice for all records",
            "No corrections needed — 0 mismatches found"
        )

    print(f"TotalPrice mismatches found & fixed: {n_mismatch}")


TotalPrice mismatches found & fixed: 0


In [31]:
# Cell 12 — Verification Gate: prove 0% duplicate IDs and 0% bad date formats
def verify_iso_date(value):
    try:
        datetime.strptime(str(value), "%Y-%m-%d")
        return True
    except ValueError:
        return False

dup_id_rate = df["OrderID"].duplicated().sum() / len(df) * 100
bad_date_rate = (~df["Date"].apply(verify_iso_date)).sum() / len(df) * 100 if "Date" in df.columns else 0
remaining_nulls = df.isnull().sum().sum()

print(f"Duplicate ID error rate:   {dup_id_rate:.2f}%  (target: 0%)")
print(f"Bad date format error rate: {bad_date_rate:.2f}%  (target: 0%)")
print(f"Remaining missing values:  {remaining_nulls}  (target: 0)")

gate_passed = (dup_id_rate == 0) and (bad_date_rate == 0) and (remaining_nulls == 0)
print("\nVERIFICATION GATE:", "PASSED ✅" if gate_passed else "FAILED ❌")

assert dup_id_rate == 0, "Verification Gate failed: duplicate OrderIDs remain"
assert bad_date_rate == 0, "Verification Gate failed: badly formatted dates remain"
assert remaining_nulls == 0, "Verification Gate failed: missing values remain"


Duplicate ID error rate:   0.00%  (target: 0%)
Bad date format error rate: 0.00%  (target: 0%)
Remaining missing values:  0  (target: 0)

VERIFICATION GATE: PASSED ✅


In [32]:
# Cell 13 — Build the Change Log table
change_log_df = pd.DataFrame(change_log)
change_log_df.index += 1
change_log_df.index.name = "No."
change_log_df


,Change ID,Description,Impact,Status
No.,,,,
1,CR001,Imputed 'CouponCode' missing values with expli...,Preserved 309 records (avoided deletion),Resolved
2,CR_DEDUPE,Removed full-row duplicates and collapsed dupl...,Removed 0 duplicate record(s); 1200 unique rec...,Resolved
3,CR_TEXT_STD,Trimmed whitespace and applied consistent Titl...,Corrected formatting on 0 cell(s),Resolved
4,CR_DATE_ISO,Converted 'Date' column to ISO 8601 format (YY...,Standardized 1200 records; 0 unparseable date(...,Resolved
5,CR_NUMERIC_PRECISION,"Rounded UnitPrice, TotalPrice to 2 decimal pla...",Applied to 1200 records,Resolved
6,CR_LOGIC_TOTALPRICE,Verified 'TotalPrice' equals Quantity x UnitPr...,No corrections needed — 0 mismatches found,Resolved


In [33]:
# Cell 14 — Export the cleaned "gold standard" dataset
CLEAN_FILE = "Cleaned_Dataset_for_Data_Analytics.xlsx"
df.to_excel(CLEAN_FILE, index=False)

CHANGE_LOG_FILE = "Change_Log.xlsx"
change_log_df.to_excel(CHANGE_LOG_FILE)

print(f"Cleaned dataset saved to: {CLEAN_FILE}")
print(f"Change log saved to: {CHANGE_LOG_FILE}")
print(f"\nFinal shape: {df.shape}")
df.head()


Cleaned dataset saved to: Cleaned_Dataset_for_Data_Analytics.xlsx
Change log saved to: Change_Log.xlsx

Final shape: (1200, 14)


,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


In [34]:
!pip install reportlab

ERROR: Could not find a version that satisfies the requirement reportlab (from versions: none)

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\SAIM\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for reportlab


In [36]:
import sys
!{sys.executable} -m pip install reportlab

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.5/2.0 MB 306.8 kB/s eta 0:00:05
   ---------- ----------------------------- 0.5/2.0 MB 306.8 kB/s eta 0:00:05
   ---------- ----------------------------- 0.5/2.0 MB 306.8 kB/s eta 0:00:05
   ----


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\SAIM\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [39]:
# Cell 15 — Generate a formal PDF Change Log artifact
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors

pdf_path = "Change_Log.pdf"
styles = getSampleStyleSheet()
story = []

story.append(Paragraph(": Data Cleaning & Preparation — Change Log", styles["Title"]))
story.append(Paragraph(" Industrial Training Kit | Batch 2026", styles["Normal"]))
story.append(Spacer(1, 16))

# Paragraph styles so long text wraps INSIDE each column instead of overlapping
header_style = ParagraphStyle(
    name="HeaderText",
    fontSize=9,
    leading=11,
    textColor=colors.white,
    fontName="Helvetica-Bold",
)

cell_style = ParagraphStyle(
    name="CellText",
    fontSize=8,
    leading=10,
    wordWrap="CJK",  # forces wrapping even on long unbroken text
)

# Build table rows using Paragraph objects instead of plain strings
table_data = [[Paragraph(h, header_style) for h in ["Change ID", "Description", "Impact", "Status"]]]

for row in change_log:
    table_data.append([
        Paragraph(str(row["Change ID"]), cell_style),
        Paragraph(str(row["Description"]), cell_style),
        Paragraph(str(row["Impact"]), cell_style),
        Paragraph(str(row["Status"]), cell_style),
    ])

log_table = Table(table_data, colWidths=[65, 230, 160, 55], repeatRows=1)
log_table.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#6E8B5E")),
    ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
    ("FONTSIZE", (0, 0), (-1, -1), 8),
    ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
    ("VALIGN", (0, 0), (-1, -1), "TOP"),
    ("LEFTPADDING", (0, 0), (-1, -1), 5),
    ("RIGHTPADDING", (0, 0), (-1, -1), 5),
    ("TOPPADDING", (0, 0), (-1, -1), 4),
    ("BOTTOMPADDING", (0, 0), (-1, -1), 4),
    ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#F2F5EF")]),
]))
story.append(log_table)

story.append(Spacer(1, 16))
story.append(Paragraph(
    f"Verification Gate: Duplicate ID error rate = {dup_id_rate:.2f}% | "
    f"Bad date format rate = {bad_date_rate:.2f}% | Remaining nulls = {remaining_nulls}",
    styles["Normal"]
))

doc = SimpleDocTemplate(pdf_path, pagesize=letter)
doc.build(story)

print(f"PDF change log generated: {pdf_path}")

PDF change log generated: Change_Log.pdf
